In [ ]:
#this code is based on the previous scraper for LPAs - this one however, instead of scraping pdf documents nested in the "Documents" tab,
#goes into the "Comments" tab, where some of the LPAs display the comments directly

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import os
import random
from urllib.parse import urljoin, urlparse
import certifi
import glob
import shutil

In [ ]:
LPAs_df_smaller = pd.read_csv("LPAs_smaller_df.txt")
LPAs_df_smaller.head()

,Country,Planning_Authority,Technology_Type,Storage_Type,Planning_Application_Reference,URLS_ADVANCED
0,Scotland,Aberdeen City,Battery,Stand-alone Storage,210665/DPP,https://publicaccess.aberdeencity.gov.uk/onlin...
1,Scotland,Aberdeen City,Battery,Stand-alone Storage,220026/DPP,https://publicaccess.aberdeencity.gov.uk/onlin...
2,Scotland,Aberdeen City,Battery,Stand-alone Storage,231336/DPP,https://publicaccess.aberdeencity.gov.uk/onlin...
3,Scotland,Aberdeen City,Battery,Stand-alone Storage,240614/DPP,https://publicaccess.aberdeencity.gov.uk/onlin...
4,Scotland,Aberdeen City,Battery,Stand-alone Storage,231134/DPP,https://publicaccess.aberdeencity.gov.uk/onlin...


In [ ]:
LPAs_df_smaller.shape

(189, 6)

In [ ]:
LPAs_df_smaller["Planning_Application_Reference"].is_unique

False

In [ ]:
session = requests.Session()

session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-GB,en;q=0.5'
})

In [ ]:
#some of the councils had broken certificates, i had to download the intermediate one and insert it for the scraper to work
CERT_DIR = "LPAs_intermediate_certs"
CUSTOM_CA_BUNDLE = "custom_ca_bundle.crt"

# 1) start from certifi bundle
shutil.copyfile(certifi.where(), CUSTOM_CA_BUNDLE)

for fp in sorted(glob.glob(os.path.join(CERT_DIR, "*"))):
    with open(fp, "rb") as inc, open(CUSTOM_CA_BUNDLE, "ab") as out:
        out.write(b"\n")
        out.write(inc.read())
print("Custom CA bundle created:", CUSTOM_CA_BUNDLE)


SSL_BROKEN_CHAIN_HOSTS = {
    "idoxwam.dundeecity.gov.uk",
    "publicaccess.glasgow.gov.uk",
    "publicaccess.southlanarkshire.gov.uk",
    "ercbuildingstandards.eastrenfrewshire.gov.uk",
    "www.eplanning.north-ayrshire.gov.uk",
    "publicaccess.south-ayrshire.gov.uk",
    "planning.inverclyde.gov.uk",
    "pa.eastlothian.gov.uk",
    "planning.westlothian.gov.uk",
    "planning.cne-siar.gov.uk"
}

Custom CA bundle created: custom_ca_bundle.crt


In [ ]:
def verify_for(url: str):
    host = urlparse(url).netloc.lower()
    if host in SSL_BROKEN_CHAIN_HOSTS:
        return CUSTOM_CA_BUNDLE
    return True

#wrappers for session calls
def sget(session, url, **kwargs):
    kwargs.setdefault("verify", verify_for(url))
    return session.get(url, **kwargs)

def spost(session, url, **kwargs):
    kwargs.setdefault("verify", verify_for(url))
    return session.post(url, **kwargs)

In [ ]:
def get_base_url(url):
    parsed = urlparse(url)
    origin = f"{parsed.scheme}://{parsed.netloc}"
    first_segment = parsed.path.split("/")[1]
    return f"{origin}/{first_segment}/"

In [ ]:
def from_advanced_to_project(session, advanced_url, reference):
#Loads advanced search page and returns the value of the hidden _csrf input

#getting to the advanced search page
    r = sget(session, advanced_url, timeout =30)

    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
#extracting crsf
    csrf = soup.select_one('input[name="_csrf"]')
    if not csrf or not csrf.get("value"):
        raise RuntimeError("No _csrf found on advanced search page")
    csrf_value = csrf["value"]


#post url (simulating clicking the 'search' button)
    base_url = get_base_url(advanced_url)
    post_url = urljoin(base_url, "advancedSearchResults.do?action=firstPage")

    payload = {
        "_csrf": csrf_value,
        "searchCriteria.reference": reference,
        "caseAddressType": "Application",
        "searchType": "Application",
        }

    r_results = spost(session, post_url, data=payload, timeout=30)
    r_results.raise_for_status()

    return r_results.url, r_results.text

In [ ]:
#getting from project's mainpage to Comments
def get_comments_url(project_html, current_url):
    soup = BeautifulSoup(project_html, "html.parser")

    a = soup.select_one("a#tab_makeComment")
    if not a:
        return None
        #raise RuntimeError(Comment tab link not found")

    return urljoin(current_url, a["href"])

In [ ]:
#from comments to public comments
def get_public_comments_url(comment_html, current_url):
    soup = BeautifulSoup(comment_html, "html.parser")

    a = soup.select_one("a#subtab_neighbourComments")
    if not a:
        return None
        #raise RuntimeError(Comment tab link not found")

    return urljoin(current_url, a["href"])

In [ ]:
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse

def set_page_param(url, page_no):
    parsed = urlparse(url)
    qs = parse_qs(parsed.query)

    qs["activeTab"] = ["neighbourComments"]
    qs["neighbourCommentsPager.page"] = [str(page_no)]

    new_query = urlencode(qs, doseq=True)
    return urlunparse(parsed._replace(query=new_query))


def scrape_public_comments(session, public_comments_url, authority=None, reference=None):
    rows = []
    page_no = 1

    while True:
        page_url = set_page_param(public_comments_url, page_no)

        r = sget(session, page_url, timeout=30)
        r.raise_for_status()

        soup = BeautifulSoup(r.text, "html.parser")
        text_divs = soup.select("div.comment-text")

        if not text_divs:
            break

        for text_div in text_divs:
            comment_text = text_div.get_text(" ", strip=True)

            if "see" in comment_text.lower() and "document" in comment_text.lower() and len(comment_text.lower())<40:
                continue


            rows.append({
                "Planning Authority": authority,
                "Reference": reference,
                "Comment": comment_text
            })

        # stop if next page does not exist
        pager_links = soup.select('a[href*="neighbourCommentsPager.page="]')
        pager_pages = set()

        for a in pager_links:
            href = a.get("href", "")
            if "neighbourCommentsPager.page=" in href:
                try:
                    n = int(href.split("neighbourCommentsPager.page=")[1].split("&")[0])
                    pager_pages.add(n)
                except ValueError:
                    pass

        if (page_no + 1) not in pager_pages:
            break

        time.sleep(random.uniform(7, 12)) #pause before requesting next pg

        #for avoiding getting blocked
        if page_no % 10 == 0:
            print(f"Scraped {page_no} comment pages. Pausing for 60 seconds...")
            time.sleep(60)

        page_no += 1

    return rows

In [ ]:
def extract_project_public_comments(session, advanced_url, authority, reference):
    #advanced search -> project page
    project_url, project_html = from_advanced_to_project(session, advanced_url, reference)
    time.sleep(random.uniform(1,4))

    #project page -> comments tab
    comments_url = get_comments_url(project_html, project_url)
    if not comments_url:
        raise RuntimeError("comments_tab_missing")
    time.sleep(random.uniform(2, 4))

    #load comments tab
    r_comments = sget(session, comments_url, timeout=30)
    r_comments.raise_for_status()
    time.sleep(random.uniform(6,13))

    #comments tab -> public comments subtab
    public_comments_url = get_public_comments_url(r_comments.text, r_comments.url)
    if not public_comments_url:
        raise RuntimeError("public_comments_tab_missing")
    time.sleep(random.uniform(2, 7))

    #scrape public comments
    rows = scrape_public_comments(
        session=session,
        public_comments_url=public_comments_url,
        authority=authority,
        reference=reference
    )

    time.sleep(random.uniform(2,5))

    return rows

In [ ]:
all_comment_rows = []
banned_councils = set()
# for rerunning the code, but skipping projects already completed
completed_keys = set()

if os.path.exists("LPAs_comments_log.csv"):
    previous_log = pd.read_csv("LPsA_comments_log.csv")

    success_statuses = [
        "success",
        "no_public_comments",
        "comments_tab_missing",
        "public_comments_tab_missing"
    ]

    completed_refs = previous_log[previous_log['Status'].isin(success_statuses)].copy()
    completed_keys = set(zip(completed_refs["Planning Authority"].astype(str).str.strip(), completed_refs["Reference"].astype(str).str.strip()))
    print(f"Found {len(completed_refs)} previously completed projects. They will be skipped.")

    all_results = previous_log.to_dict('records')
else:
    all_results = []



if os.path.exists("LPAs_comments.csv"):
    try:
        previous_log = pd.read_csv("LPAs_comments.csv")
        all_comment_rows = previous_log.to_dict("records")
    except pd.errors.EmptyDataError:
        print("LPAs_comments.csv exists but is empty — starting fresh.")


for idx, row in LPAs_df_smaller.iterrows():

    # periodic save
    if idx > 0 and idx % 10 == 0:

        pd.DataFrame(all_results).drop_duplicates(
            subset=["Planning Authority", "Reference"],
            keep="last"
        ).to_csv("LPAs_comments_log.csv", index=False, encoding="utf-8")

        pd.DataFrame(all_comment_rows).drop_duplicates(
            subset=["Planning Authority", "Reference", "Comment"]
        ).to_csv("LPAs_comments.csv", index=False, encoding="utf-8")


    advanced_url = str(row["URLS_ADVANCED"]).strip()
    project_ref = str(row["Planning_Application_Reference"]).strip()
    authority = str(row["Planning_Authority"]).strip()

    key = (authority, project_ref)


    # skip councils already blocked by Cloudflare
    if authority in banned_councils:
        print(f"{authority} is banned (Cloudflare detected earlier). Skipping.")
        all_results.append({
            "Planning Authority": authority,
            "Reference": project_ref,
            "Status": "skipped_after_cloudflare",
            "Error": ""
        })
        continue


    if key in completed_keys:
        print(f"Project {project_ref} by {authority} processed in a previous run. Skipping.")
        continue

    print(f"[{idx+1}/{len(LPAs_df_smaller)}] {project_ref}")


    try:
        rows = extract_project_public_comments(
            session=session,
            advanced_url=advanced_url,
            authority=authority,
            reference=project_ref
        )

        if rows:
            all_comment_rows.extend(rows)
            all_results.append({
                "Planning Authority": authority,
                "Reference": project_ref,
                "Status": "success",
                "Error": ""
            })
            completed_keys.add(key)
            print(f"Scraped {len(rows)} comments.")
        else:
            all_results.append({
                "Planning Authority": authority,
                "Reference": project_ref,
                "Status": "no_public_comments",
                "Error": ""
            })
            completed_keys.add(key)
            print("No public comments found.")

        time.sleep(random.uniform(8, 12))

    except requests.exceptions.HTTPError as e:
        if e.response is not None:
            text = (e.response.text or "").lower()
            if e.response.status_code in (403, 429) and (
                "cloudflare" in text
                or "checking your browser" in text
                or "security verification" in text
                or "cf-ray" in e.response.headers
            ):
                print(f"403 Error! {authority} blocked by Cloudflare.")
                all_results.append({
                    "Planning Authority": authority,
                    "Reference": project_ref,
                    "Status": "cloudflare_block",
                    "Error": repr(e)
                })
                banned_councils.add(authority)
                continue


        all_results.append({
            "Planning Authority": authority,
            "Reference": project_ref,
            "Status": f"http_error",
            "Error": repr(e)
        })
        print(f"HTTP error processing reference {project_ref}: {e}")
        print("Pausing for 60 seconds.")
        time.sleep(60)
        continue

    except RuntimeError as e:
        msg = str(e)

        if msg == "comments_tab_missing":
            status = "comments_tab_missing"
        elif msg == "public_comments_tab_missing":
            status = "public_comments_tab_missing"
        else:
            status = "runtime_error"

        all_results.append({
            "Planning Authority": authority,
            "Reference": project_ref,
            "Status": status,
            "Error": msg
        })

        if status in ["comments_tab_missing", "public_comments_tab_missing"]:
            completed_keys.add(key)
        print(f"{status} for reference {project_ref}")
        time.sleep(random.uniform(3,7))
        continue

    except Exception as e:
        all_results.append({
            "Planning Authority": authority,
            "Reference": project_ref,
            "Status": "exception",
            "Error": repr(e)
        })
        print(f"Error processing reference {project_ref}: {e}")
        time.sleep(random.uniform(2, 10))
        continue

# final save
df_comments = pd.DataFrame(all_comment_rows).drop_duplicates(
    subset=["Planning Authority", "Reference", "Comment"]
)
df_results = pd.DataFrame(all_results).drop_duplicates(
    subset=["Planning Authority", "Reference"],
    keep="last"
)

df_comments.to_csv("LPAs_comments.csv", index=False, encoding="utf-8")
df_results.to_csv("LPAs_comments_log.csv", index=False, encoding="utf-8")

Found 178 previously completed projects. They will be skipped.
Project 210665/DPP by Aberdeen City processed in a previous run. Skipping.
Project 220026/DPP by Aberdeen City processed in a previous run. Skipping.
Project 231336/DPP by Aberdeen City processed in a previous run. Skipping.
Project 240614/DPP by Aberdeen City processed in a previous run. Skipping.
Project 231134/DPP by Aberdeen City processed in a previous run. Skipping.
Project 241197/DPP by Aberdeen City processed in a previous run. Skipping.
Project 240313/DPP by Aberdeen City processed in a previous run. Skipping.
Project APP/2018/0488 by Aberdeenshire processed in a previous run. Skipping.
Project APP/2018/0525 by Aberdeenshire processed in a previous run. Skipping.
Project APP/2018/0526 by Aberdeenshire processed in a previous run. Skipping.
Project APP/2018/2702 by Aberdeenshire processed in a previous run. Skipping.
Project APP/2019/0373 by Aberdeenshire processed in a previous run. Skipping.
Project ENQ/2020/0261 